<center><h1>Raj Panchasara</h1></center>
<center><h1>23010101185</h1></center><br>
<center><h2>Implement Algorithm without use of Library</h2></center>

# Week 4 Task :

## Import library

In [1]:
import numpy as np
import pandas as pd

## Load Clean Dataset

In [2]:
df = pd.read_csv("cardio_train_clean.csv")

X = df.drop("cardio", axis=1).values
y = df["cardio"].values

## Train–Test Split

In [3]:
np.random.seed(18)
indices = np.random.permutation(len(X))

split = int(0.8 * len(X))
train_idx, test_idx = indices[:split], indices[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


## Gini Impurity Function

In [4]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    prob = counts / counts.sum()
    return 1 - np.sum(prob ** 2)


## Decision Tree

In [5]:
def best_split(X, y):
    best_feature, best_threshold, best_gini = None, None, float("inf")

    n_features = X.shape[1]
    features = np.random.choice(n_features, int(np.sqrt(n_features)), replace=False)

    for feature in features:
        thresholds = np.unique(X[:, feature])
        for t in thresholds:
            left = y[X[:, feature] <= t]
            right = y[X[:, feature] > t]

            if len(left) == 0 or len(right) == 0:
                continue

            g = (len(left) * gini(left) + len(right) * gini(right)) / len(y)

            if g < best_gini:
                best_feature, best_threshold, best_gini = feature, t, g

    return best_feature, best_threshold


## Build Decision Tree

In [6]:
def build_tree(X, y, depth=0, max_depth=5):
    if len(np.unique(y)) == 1 or depth == max_depth:
        return np.bincount(y).argmax()

    feature, threshold = best_split(X, y)
    if feature is None:
        return np.bincount(y).argmax()

    left_idx = X[:, feature] <= threshold
    right_idx = X[:, feature] > threshold

    return {
        "feature": feature,
        "threshold": threshold,
        "left": build_tree(X[left_idx], y[left_idx], depth + 1),
        "right": build_tree(X[right_idx], y[right_idx], depth + 1)
    }


## Tree Prediction

In [7]:
def predict_tree(tree, x):
    if not isinstance(tree, dict):
        return tree

    if x[tree["feature"]] <= tree["threshold"]:
        return predict_tree(tree["left"], x)
    else:
        return predict_tree(tree["right"], x)


## Random Forest Training

In [8]:
def train_random_forest(X, y, n_trees=10):
    forest = []

    for _ in range(n_trees):
        idx = np.random.choice(len(X), len(X), replace=True)
        tree = build_tree(X[idx], y[idx])
        forest.append(tree)

    return forest


## Random Forest Prediction

In [9]:
def predict_forest(forest, X):
    predictions = []

    for x in X:
        votes = [predict_tree(tree, x) for tree in forest]
        predictions.append(np.bincount(votes).argmax())

    return np.array(predictions)


## Train & Evaluate Random Forest

In [10]:
forest = train_random_forest(X_train, y_train, n_trees=10)

y_pred = predict_forest(forest, X_test)

accuracy = np.mean(y_pred == y_test)
print("Random Forest Accuracy (No Library):", accuracy)


Random Forest Accuracy (No Library): 0.7128229741620671


In [11]:
print("Random Forest(No Library) : ", accuracy*100)

Random Forest(No Library) :  71.2822974162067


## Load the saved model

In [13]:
import joblib

try:
    model = joblib.load("cardio_model.pkl") 
    print("Model loaded successfully!")
    
    sample_data = [[50, 1, 160, 60, 120, 80, 1, 1, 0, 0, 1]] 
    prediction = model.predict(sample_data)
    print("Test Prediction:", prediction)

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: Ensure you saved the MODEL object (e.g., 'knn'), not a list like '[knn, acc]'.")

Model loaded successfully!
Error loading model: 'list' object has no attribute 'predict'
Tip: Ensure you saved the MODEL object (e.g., 'knn'), not a list like '[knn, acc]'.


In [14]:
type(model)

list